# Analysis of experiment #12: CFCP on closed neighborhoods for DIMAC instances

## Read data

In [1]:
# Number of vertices and edges in the hypergraphs
import pandas as pd
hypergraphs={
    "name": ['david','huck','jean','myciel3','myciel4','myciel5','queen5_5','queen6_6','queen7_7'],
    "n_H": [87,74,80,11,23,47,25,36,49],
    "m_H": [74,49,66,11,23,47,25,36,49]
}
df_hypergraphs = pd.DataFrame(hypergraphs)
print(df_hypergraphs)

       name  n_H  m_H
0     david   87   74
1      huck   74   49
2      jean   80   66
3   myciel3   11   11
4   myciel4   23   23
5   myciel5   47   47
6  queen5_5   25   25
7  queen6_6   36   36
8  queen7_7   49   49


In [2]:
# Read csv

import pandas as pd
import numpy as np

df = pd.read_csv("stats12.csv")

# Add column with the density of the graph
df["density"] = df.apply(lambda row: 2 * row.nedges / (row.nvertices * (row.nvertices - 1)), axis=1)

# Join the hypergraph data to the main dataframe
df = df.join(df_hypergraphs.set_index("name"), on="instance", how="left")

# Remove final _ in solver name
df['solver'] = df['solver'].str.rstrip('_')

# Choose solvers to keep
df = df[df['solver'].isin(['byp', 'gurobi'])]

# Change solver name
df['solver'] = df['solver'].replace('byp', 'b&p')

# Sort values by instance and solver
df = df.sort_values(by=["instance", "solver"]).reset_index(drop=True)

# Unify TIME_EXCEEDED values
df = df.replace({'TIME_EXCEEDED_PR': 'TIME_EXCEEDED', 'TIME_EXCEEDED_LP': 'TIME_EXCEEDED'}) 

# Remove the heuristic time from the total time
df.time = df.time - df.initialHeurTime

# Format timelimit
df.loc[df.state == 'TIME_EXCEEDED', 'time'] = 3600

# Format number of nodes for cplex
df.loc[(df.solver == 'cplex') & (df.nodes == 0), 'nodes'] = 1

print(list(df.columns))
df.head()

['instance', 'solver', 'run', 'nvertices', 'nedges', 'nP', 'nQ', 'nvars', 'ncons', 'state', 'terminationReason', 'time', 'nodes', 'nodesLeft', 'lb', 'ub', 'gap', 'initialHeurValue', 'initialHeurTime', 'initialSemigreedyIters', 'nNodesInt', 'nNodesFrac', 'nNodesGcp', 'nNodesTrivial', 'nNodesInfeas', 'nNodesInfeasPrepro', 'nNodesInfeasCheck', 'nNodesInfeasAux', 'gcpAvgTime', 'nsol', 'nsolHeur', 'nsolLR', 'nsolGCP', 'nsolTrivial', 'ninitSol', 'ninitDummy', 'ninit', 'rootNVertices', 'rootNEdges', 'rootNP', 'rootNQ', 'rootlb', 'rootub', 'rootHeurTime', 'rootFeasTime', 'rootCgTime', 'rootNCalls', 'rootNCallsPool', 'rootNCallsHeur', 'rootNCallsMwis1', 'rootNCallsMwis2', 'rootNCallsExact', 'rootNCols', 'rootNColsPool', 'rootNColsHeur', 'rootNColsMwis1', 'rootNColsMwis2', 'rootNColsExact', 'rootTime', 'rootTimePool', 'rootTimeHeur', 'rootTimeMwis1', 'rootTimeMwis2', 'rootTimeExact', 'otherNodesHeurTime', 'otherNodesFeasNCalls', 'otherNodesFeasTime', 'otherNodesNCalls', 'otherNodesNCallsPool', '

,instance,solver,run,nvertices,nedges,nP,nQ,nvars,ncons,state,...,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,density,n_H,m_H
0,david,b&p,0,791,195799,74,87,-1,-1,OPTIMAL,...,0.0,23.747600,0.085624,4.590280,16.968300,2.103360,0.0,0.626667,87,74
1,david,gurobi,0,791,195799,74,87,2376,669959,OPTIMAL,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.626667,87,74
2,huck,b&p,0,455,53611,49,74,-1,-1,OPTIMAL,...,0.0,0.231217,0.000367,0.020268,0.003686,0.206897,0.0,0.519059,74,49
3,huck,gurobi,0,455,53611,49,74,1824,265997,OPTIMAL,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.519059,74,49
4,jean,b&p,0,477,45782,66,80,-1,-1,OPTIMAL,...,0.0,1.229230,0.003958,0.300493,0.668337,0.256441,0.0,0.403273,80,66


In [3]:
# pivot table by solver
df2 = df.pivot(index=["instance", "nvertices", "density", "nP", "nQ", "initialHeurValue"], 
                columns="solver", 
                values=["time", "nodes", "lb", "ub", "state"],
                ).reset_index()
print(list(df2.columns))
df2.head()

[('instance', ''), ('nvertices', ''), ('density', ''), ('nP', ''), ('nQ', ''), ('initialHeurValue', ''), ('time', 'b&p'), ('time', 'gurobi'), ('nodes', 'b&p'), ('nodes', 'gurobi'), ('lb', 'b&p'), ('lb', 'gurobi'), ('ub', 'b&p'), ('ub', 'gurobi'), ('state', 'b&p'), ('state', 'gurobi')]


instance nvertices   density  nP  nQ initialHeurValue     time  \
solver                                                            b&p   
0         david       791  0.626667  74  87              3.0  155.486   
1          huck       455  0.519059  49  74              4.0    8.188   
2          jean       477  0.403273  66  80              4.0   93.532   
3       myciel3        51  0.560784  11  11              2.0  0.00775   
4       myciel4       165  0.500665  23  23              3.0   3.4779   

                nodes          lb          ub           state           
solver   gurobi   b&p gurobi  b&p gurobi  b&p gurobi      b&p   gurobi  
0       330.427     5      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL  
1       123.352     7      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL  
2       164.232    55      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL  
3       0.10856     1      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL  
4        7.2343    19      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL

In [7]:
df3 = df2[[('instance',''), ('nvertices',''), ('density',''), ('nP',''), ('nQ',''), ('initialHeurValue',''), 
           ('time','b&p'), ('time','gurobi'), ('nodes','b&p'), ('nodes','gurobi'), 
           ('lb','b&p'), ('lb','gurobi'), ('ub','b&p'), ('ub','gurobi')]]
colNames = pd.MultiIndex.from_tuples([('instance',''), ('|V|',''), ('density',''), ('n',''), ('m',''), ('hval',''), 
           ('time (s)','b&p'), ('time (s)','gurobi'), ('nodes','b&p'), ('nodes','gurobi'), 
           ('lb','b&p'), ('lb','gurobi'), ('ub','b&p'), ('ub','gurobi')])
df3.columns = colNames
df3[('density','')] = df3[('density','')].round(2)
df3[('hval','')] = df3[('hval','')].astype(int)
df3[('nodes','b&p')] = df3[('nodes','b&p')].astype(int)
df3[('nodes','gurobi')] = df3[('nodes','gurobi')].astype(int)
df3[('lb','b&p')] = df3[('lb','b&p')].round(2)
df3[('lb','gurobi')] = df3[('lb','gurobi')].round(2)
df3[('ub','b&p')] = df3[('ub','b&p')].astype(int)
df3[('ub','gurobi')] = df3[('ub','gurobi')].astype(int)

# Specific formatting
df3[('time (s)','b&p')] = df2.apply(lambda row: "tilim" if row[('state','b&p')] == 'TIME_EXCEEDED' 
                                    else ("memlim" if row[('state','b&p')] == 'MEM_EXCEEDED' 
                                          else "{:.1f}".format(row[('time','b&p')])), axis=1)
df3[('time (s)','gurobi')] = df2.apply(lambda row: "tilim" if row[('state','gurobi')] == 'TIME_EXCEEDED' 
                                       else ("memlim" if row[('state','gurobi')] == 'MEM_EXCEEDED' 
                                             else "{:.1f}".format(row[('time','gurobi')])), axis=1)
df3[('lb','b&p')] = df3[('lb','b&p')].apply(lambda x: "{:.1f}".format(x) if x >= 0 else "--")
df3[('lb','gurobi')] = df3[('lb','gurobi')].apply(lambda x: "{:.1f}".format(x) if x >= 0 else "--")

df3

instance   |V| density   n   m hval time (s)         nodes          lb  \
                                            b&p  gurobi   b&p gurobi  b&p   
0     david   791    0.63  74  87    3    155.5   330.4     5      1  2.0   
1      huck   455    0.52  49  74    4      8.2   123.4     7      1  2.0   
2      jean   477    0.40  66  80    4     93.5   164.2    55      1  2.0   
3   myciel3    51    0.56  11  11    2      0.0     0.1     1      1  2.0   
4   myciel4   165    0.50  23  23    3      3.5     7.2    19      1  2.0   
5   myciel5   519    0.45  47  47    3    202.7   239.1    49      1  2.0   
6  queen5_5   345    0.75  25  25    2      0.6    66.3     1      1  2.0   
7  queen6_6   616    0.69  36  36    2      6.1   432.1     1      1  2.0   
8  queen7_7  1001    0.63  49  49    4    tilim  memlim     1      1   --   

          ub         
  gurobi b&p gurobi  
0    2.0   2      2  
1    2.0   2      2  
2    2.0   2      2  
3    2.0   2      2  
4    2.0   2      2  
5    2.0   2      2  
6    2.0   2      2  
7    2.0   2      2  
8    1.0   4      4

In [5]:
df3.to_latex("table12.tex", index=False, float_format="%.1f", na_rep="--")